In [1]:

!pip install PyPDF2 langchain-text-splitters python-docx


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [langchain-text-splitters]


In [2]:

!pip install -q PyPDF2 langchain-text-splitters


In [3]:

from google.colab import files

uploaded = files.upload()

print("Uploaded files:", list(uploaded.keys()))


Saving 2402.19473v6.pdf to 2402.19473v6.pdf
Uploaded files: ['2402.19473v6.pdf']


In [4]:
!pip install -q pymupdf langchain-text-splitters pytesseract pillow
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [6]:
from google.colab import files

if "uploaded" in globals() and uploaded:
    print("Previously uploaded files:")
    for filename in uploaded:
        print("-", filename)
else:
    print("No uploaded files found. Please upload your PDF.")
    uploaded = files.upload()

Previously uploaded files:
- 2402.19473v6.pdf


In [7]:
!pip install -q pymupdf easyocr langchain-text-splitters

In [8]:
import fitz
import easyocr
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("PyMuPDF: Working")
print("EasyOCR: Working")
print("Text splitter: Working")

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


PyMuPDF: Working
EasyOCR: Working
Text splitter: Working


In [9]:
import pymupdf
import easyocr
import numpy as np
from PIL import Image
from io import BytesIO
import re

# Use the PDF already uploaded in your notebook
pdf_path = list(uploaded.keys())[0]

# Initialize EasyOCR (English)
reader = easyocr.Reader(['en'], gpu=False)

doc = pymupdf.open(pdf_path)
extracted_pages = []

for page_num, page in enumerate(doc, start=1):
    print(f"Processing page {page_num}/{len(doc)}...")

    # 1. Extract selectable PDF text
    page_text = page.get_text("text").strip()

    # 2. OCR the full page if selectable text is missing
    if len(page_text) < 30:
        pix = page.get_pixmap(matrix=pymupdf.Matrix(2, 2))
        page_image = Image.open(BytesIO(pix.tobytes("png"))).convert("RGB")
        page_array = np.array(page_image)
        ocr_results = reader.readtext(page_array, detail=0)
        ocr_text = "\n".join(ocr_results)

        if ocr_text.strip():
            page_text = ocr_text
            method = "easyocr_page"
        else:
            method = "pymupdf"
    else:
        method = "pymupdf"

    # 3. OCR embedded images to capture text inside figures
    image_texts = []

    for img_info in page.get_images(full=True):
        xref = img_info[0]

        try:
            image_data = doc.extract_image(xref)
            image = Image.open(
                BytesIO(image_data["image"])
            ).convert("RGB")

            image_results = reader.readtext(
                np.array(image), detail=0
            )

            if image_results:
                image_texts.append("\n".join(image_results))

        except Exception as error:
            print(f"Skipping an image on page {page_num}: {error}")

    # Save page text and OCR text separately to avoid losing provenance
    extracted_pages.append({
        "page": page_num,
        "text": page_text,
        "extraction_method": method,
        "image_ocr_text": image_texts
    })

doc.close()

print("\nExtraction complete!")
print("Pages processed:", len(extracted_pages))
print("Pages with text:", sum(bool(p["text"]) for p in extracted_pages))
print(
    "Images with recognized text:",
    sum(len(p["image_ocr_text"]) for p in extracted_pages)
)

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% CompleteProcessing page 1/22...
Processing page 2/22...
Processing page 3/22...
Processing page 4/22...
Processing page 5/22...
Processing page 6/22...
Processing page 7/22...
Processing page 8/22...
Processing page 9/22...
Processing page 10/22...
Processing page 11/22...
Processing page 12/22...
Processing page 13/22...
Processing page 14/22...


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Processing page 15/22...
Processing page 16/22...
Processing page 17/22...
Processing page 18/22...
Processing page 19/22...
Processing page 20/22...
Processing page 21/22...
Processing page 22/22...

Extraction complete!
Pages processed: 22
Pages with text: 22
Images with recognized text: 6


In [18]:

import re
import json
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    separators=["\n\n", "\n", ". ", " ", ""]
)

text_chunks = []
chunk_id = 0

for page_data in extracted_pages:
    page_num = page_data["page"]

    # Process normal PDF text separately
    sections = []

    if page_data["text"].strip():
        sections.append({
            "text": page_data["text"],
            "source_type": "pdf_text"
        })

    # Process text detected inside images separately
    for image_text in page_data["image_ocr_text"]:
        if image_text.strip():
            sections.append({
                "text": image_text,
                "source_type": "image_ocr"
            })

    for section in sections:
        cleaned_text = re.sub(r"[ \t]+", " ", section["text"])
        cleaned_text = re.sub(r"\n{3,}", "\n\n", cleaned_text)
        cleaned_text = cleaned_text.strip()

        if not cleaned_text:
            continue

        # Skip only very short OCR fragments, not normal PDF text
        if section["source_type"] == "image_ocr":
            if len(cleaned_text) < 20 and len(cleaned_text.split()) < 4:
                print("Skipped short OCR fragment:", repr(cleaned_text))
                continue

        chunks = splitter.split_text(cleaned_text)

        for chunk in chunks:
            text_chunks.append({
                "chunk_id": chunk_id,
                "source": pdf_path,
                "page": page_num,
                "source_type": section["source_type"],
                "text": chunk
            })
            chunk_id += 1

with open("text_chunks.json", "w", encoding="utf-8") as f:
    json.dump(text_chunks, f, ensure_ascii=False, indent=2)

print("\nChunking complete!")
print("Total chunks:", len(text_chunks))
print("PDF text chunks:", sum(c["source_type"] == "pdf_text" for c in text_chunks))
print("Image OCR chunks:", sum(c["source_type"] == "image_ocr" for c in text_chunks))
print("\nFirst chunk preview:\n")
print(text_chunks[0]["text"] if text_chunks else "No chunks generated.")
print("\nSaved updated text_chunks.json")


Skipped short OCR fragment: 'Utterseeel'
Skipped short OCR fragment: 'rSt'
Skipped short OCR fragment: 'OH\nHO\nOH'

Chunking complete!
Total chunks: 214
PDF text chunks: 211
Image OCR chunks: 3

First chunk preview:

1
Retrieval-Augmented Generation for
AI-Generated Content: A Survey
Penghao Zhao∗, Hailin Zhang∗, Qinhan Yu, Zhengren Wang, Yunteng Geng,
Fangcheng Fu†, Ling Yang, Wentao Zhang†, Jie Jiang, Bin Cui†
Abstract—Advancements in model algorithms, the growth of
foundational models, and access to high-quality datasets have
propelled the evolution of Artificial Intelligence Generated Con-
tent (AIGC). Despite its notable successes, AIGC still faces
hurdles such as updating knowledge, handling long-tail data,
mitigating data leakage, and managing high training and infer-
ence costs. Retrieval-Augmented Generation (RAG) has recently
emerged as a paradigm to address such challenges. In partic-
ular, RAG introduces the information retrieval process, which

Saved updated text_chunks.j

In [12]:
# Check how many chunks were generated
print("Total chunks:", len(text_chunks))

# Preview three chunks
for chunk in text_chunks[:3]:
    print("\n" + "=" * 50)
    print("Chunk ID:", chunk["chunk_id"])
    print("Page:", chunk["page"])
    print("Text:", chunk["text"][:300])

Total chunks: 217

Chunk ID: 0
Page: 1
Text: 1
Retrieval-Augmented Generation for
AI-Generated Content: A Survey
Penghao Zhao∗, Hailin Zhang∗, Qinhan Yu, Zhengren Wang, Yunteng Geng,
Fangcheng Fu†, Ling Yang, Wentao Zhang†, Jie Jiang, Bin Cui†
Abstract—Advancements in model algorithms, the growth of
foundational models, and access to high-qual

Chunk ID: 1
Page: 1
Text: ular, RAG introduces the information retrieval process, which
enhances the generation process by retrieving relevant objects
from available data stores, leading to higher accuracy and better
robustness. In this paper, we comprehensively review existing
efforts that integrate RAG techniques into AIGC

Chunk ID: 2
Page: 1
Text: additional enhancements methods for RAG, facilitating effective
engineering and implementation of RAG systems. Then from
another view, we survey on practical applications of RAG across
different modalities and tasks, offering valuable references for
researchers and practitioners. Furthermore, we in

In [19]:
from google.colab import files

files.download("text_chunks.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
# Check text extraction and OCR results

print("Total pages:", len(extracted_pages))
print("Total chunks:", len(text_chunks))

print("\nPages with no extracted text:")
for page in extracted_pages:
    if not page["text"].strip() and not page["image_ocr_text"]:
        print("Page", page["page"])

print("\nChunks with very little text:")
for chunk in text_chunks:
    if len(chunk["text"].strip()) < 50:
        print("Chunk", chunk["chunk_id"], ":", repr(chunk["text"]))

Total pages: 22
Total chunks: 217

Pages with no extracted text:

Chunks with very little text:
Chunk 119 : 'Utterseeel'
Chunk 120 : 'rSt'
Chunk 130 : '0\nF\n0\nH\nNH2'
Chunk 131 : 'OH\nOH\nOH\nHO\nOH'
Chunk 132 : 'OH\nHO\nOH'
Chunk 133 : '~OH\n0\nO\nNH\n0\nNH'


In [17]:
# Inspect the suspicious chunks and their neighbors

for chunk_id in [119, 120, 130, 131, 132, 133]:
    print("\n" + "=" * 60)

    for chunk in text_chunks:
        if chunk["chunk_id"] in [chunk_id - 1, chunk_id, chunk_id + 1]:
            print(
                f"\nChunk {chunk['chunk_id']} | Page {chunk['page']}\n"
                f"{chunk['text'][:500]}"
            )



Chunk 118 | Page 14
Fig. 10: Architecture of Re-AudioLDM [116] model.
Re-AudioLDM [116] retrieves relevant caption-audio pairs
with dense retriever CLAP [26] for generation. Make-An-
Audio [44] retrieves audios given text prompt, then constructs
pseudo prompts for text-to-audio diffusion model training.

Chunk 119 | Page 14
Utterseeel

Chunk 120 | Page 14
rSt


Chunk 119 | Page 14
Utterseeel

Chunk 120 | Page 14
rSt

Chunk 121 | Page 15
15
2) Audio
Captioning:
Audio
captioning,
basically
a
sequence-to-sequence task, generates natural language data
for audio data. RECAP [323] and [43] leverages dense re-
trievers, CLAP [26] and VGGish [69] respectively, to retrieve
related captions given audio data. For RECAP, captions are
included into LLM prompts, while [43] uses both audio and
retrieved captions in attention module. Other research studies
align audio modality with text to leverage advancements in
LLMs [324]–[326] for various 


Chunk 129 | Page 15
RAG, there have also been several 